In [1]:
import os
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import SystemMessage,HumanMessage
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
import json
from huggingface_hub import InferenceClient
from langchain_huggingface import ChatHuggingFace

In [ ]:
load_dotenv()
router=OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
hugging=InferenceClient(api_key='')
load_dotenv()
ollama=OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)
hf_token = os.getenv("HUGGING_FACE_API")

In [4]:
db_name='vector_db'
load_dotenv(override=True)

True

In [5]:
embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vectordatastore=Chroma(persist_directory=db_name,embedding_function=embeddings)#connect with datastore

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [24]:
retriver=vectordatastore.as_retriever()
olama_llm=ChatOllama(temperature=0,model='llama3.2')
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
llm = HuggingFaceEndpoint(repo_id = "swiss-ai/Apertus-8B-Instruct-2509",task="text-generation",huggingfacehub_api_token=hf_token,max_new_tokens=512,)
chat_hf= ChatHuggingFace(llm=llm)


# chunks reranking

In [7]:
def chunks_unranked(question):
    docs=retriver.invoke(question)
    chunks = [
        {
            "id": i,
            "text": doc.page_content
        }
        for i, doc in enumerate(docs)
    ]
    return chunks

In [8]:
def rerank(question,chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"""
The user has asked the following question:

{question}

Order all the chunks by relevance to the question, from most relevant to least relevant.

Here are the chunks:

{chunks}
"""
    messages=[
        {"role":"system","content":system_prompt},
        {"role":"user","content":user_prompt}
    ]
    response=ollama.chat.completions.create(
        model="llama3.2",
        messages=messages
    )
    reply=response.choices[0].message.content
    reply=json.loads(reply)
    return reply

In [9]:
chunks_unranked('who is lancestor')

[{'id': 0,
  'text': '**Belvedere Insurance**  \nSignature: ______________________  \nName: [Authorized Signatory]  \nTitle: [Title]  \nDate: ______________________  \n\n--- \nThis synthetic contract document outlines a fictional agreement between Insurellm and a fictional insurance client, Belvedere Insurance, which engages with the Markellm platform. The contract contains creative yet realistic terms for potential use in training and development in insurance technology scenarios.'},
 {'id': 1,
  'text': "_________________________________\n**Jonathan Park**\n**Title**: President & CEO\n**Guardian Life Partners**\n**Date**: March 1, 2025\n\n---\n\n*This contract establishes Guardian Life Partners as a strategic partner leveraging Lifellm's advanced AI underwriting and digital health integration to modernize life insurance operations.*"},
 {'id': 2,
  'text': "## Other HR Notes\n- Jordan K. Bishop has been an integral part of club initiatives, including the Insurellm Code Reviews and Fe

In [16]:
chunks=chunks_unranked('who is lancestor')
rerank("who is lancestor", chunks)# here we can see order changed and itmay give better output

[1, 2, 3, 0]

In [10]:
def fetch_ranked(question):
    chunks=chunks_unranked(question)
    order=rerank(question,chunks)
    ranked_chunks=[chunks[i] for i in order]
    return ranked_chunks

In [14]:
fetch_ranked("who is lancestor")

[{'id': 0,
  'text': '**Belvedere Insurance**  \nSignature: ______________________  \nName: [Authorized Signatory]  \nTitle: [Title]  \nDate: ______________________  \n\n--- \nThis synthetic contract document outlines a fictional agreement between Insurellm and a fictional insurance client, Belvedere Insurance, which engages with the Markellm platform. The contract contains creative yet realistic terms for potential use in training and development in insurance technology scenarios.'},
 {'id': 2,
  'text': "## Other HR Notes\n- Jordan K. Bishop has been an integral part of club initiatives, including the Insurellm Code Reviews and Feedback Group, providing peer support.\n- Active participant in the company's Diversity and Inclusion committee, promoting a positive work culture.\n- Jordan has expressed interest in professional development courses, particularly those focused on modern web technologies, which are being considered for sponsorship by Insurellm.\n- Engaged in a 6-month perform

# now putting them together

In [28]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
dont give the context just give the answer.
Rules:
- Give a direct, concise answer.
- Usually answer in 1-3 sentences.
- Do not explain your reasoning.
- Do not show your analysis.
- Do not output <think> or </think>.
- Do not mention the retrieval process.
- If the answer is not in the context, say you don't know.
- Never invent information.
Context:
{context}
"""

In [ ]:
def answer_question(question,history):
    context=fetch_ranked(question)
    system_prompt=SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response=chat_hf.invoke([SystemMessage(content=system_prompt),HumanMessage(content=question)])
    #response=ollam_chat.invoke([SystemMessage(content=system_prompt),HumanMessage(content=question)])
    return response.content

## advantages of reranking is that it can answer very detailed answers also

In [31]:
print(answer_question('who is the ceo?',[]))

Jennifer Rodriguez.


# qwery rewriting

In [33]:
def qwery_rewrit(question,history):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
    You are in a conversation with a user, answering questions about the company Insurellm.
    You are about to look up information in a Knowledge Base to answer the user's question.
    
    This is the history of your conversation so far with the user:
    {history}
    
    And this is the user's current question:
    {question}
    
    Respond only with a short, refined question that you will use to search the Knowledge Base.
    It should be a VERY short specific question most likely to surface content. Focus on the question details.
    IMPORTANT: Respond ONLY with the precise knowledgebase query, nothing else.
    """
    response = router.chat.completions.create(model='openrouter/free', messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content
    

In [34]:
qwery_rewrit('who is the ceo?',[])

'Who is the CEO of Insurellm?'

In [69]:
def fetch_ranked(question):
    rewrite_question=qwery_rewrit(question,[])
    chunks=chunks_unranked(question)
    order=rerank(question,chunks)
    ranked_chunks=[chunks[i] for i in order]
    return ranked_chunks

In [67]:
def answer_question(question,history):
    context=fetch_ranked(question)
    system_prompt=SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response=chat_hf.invoke([SystemMessage(content=system_prompt),HumanMessage(content=question)])
    #response=ollam_chat.invoke([SystemMessage(content=system_prompt),HumanMessage(content=question)])
    return response.content

In [65]:
answer_question("what are the highlights of Alex Thomson?",[])

'Consistently maintained a 30-minute response time to inbound leads; successfully coordinated webinars for product launches that attracted over 2,000 potential customers.'

# query expansion

* here we will merge questions chunks and rewritten questions chunk

In [114]:
def merge_chunks(old_chunks, new_chunks):
    merged = old_chunks[:]
    existing = [chunk.page_content for chunk in old_chunks]
    for chunk in new_chunks:
        if chunk.page_content not in existing:# if not in old chunks
            merged.append(chunk)
    return merged

In [82]:
def chunks_unranked(question):
    docs=retriver.invoke(question)
    return docs

In [111]:
def rerank(question,chunks):
    system_prompt = """
You are a document re-ranker.

You are given a user question and a list of document chunks.
Each chunk has a numerical ID.

Rank the chunks from most relevant to least relevant.

IMPORTANT:
- Return ONLY a JSON array of integers.
- The integers must be the Chunk IDs provided in the prompt.
- Do NOT return UUIDs.
- Do NOT return document IDs.
- Do NOT return chunk text.
- Do NOT include explanations.
- Do NOT use markdown.

Example:
[2, 0, 3, 1]
"""
    user_prompt = f"""
The user has asked the following question:

{question}

Order all the chunks by relevance to the question, from most relevant to least relevant.

Here are the chunks:

{chunks}
"""
    messages=[
        {"role":"system","content":system_prompt},
        {"role":"user","content":user_prompt}
    ]
    response=ollama.chat.completions.create(
        model="llama3.2",
        messages=messages
    )
    reply=response.choices[0].message.content
    order = json.loads(reply)
    return order

In [ ]:
def answer_question(question,history):
    chunk1=chunks_unranked(question)
    rewrite=qwery_rewrit(question,history)
    chunk2=chunks_unranked(rewrite)# re written question
    merged_chunks=merge_chunks(chunk1,chunk2)# merging them
    order=rerank(question,merged_chunks)
    ranked_chunks = [merged_chunks[i] for i in order]
    context = "\n\n".join(chunk.page_content for chunk in ranked_chunks)
    system_prompt=SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response=chat_hf.invoke([SystemMessage(content=system_prompt),HumanMessage(content=question)])
    return response.content

In [113]:
answer_question("who is Robert Chen?",[])

'Robert Chen is a Senior Full Stack Engineer at Insurellm with a current salary of $152,000, based in San Francisco, California, and has been in this role since January 2016.'